# Model Serving in Machine Learning

Training a model is only half the job — **model serving** is how you turn a trained model into something users and systems can actually call. This note covers the practical serving workflow: serializing your model, wrapping it in an API, building quick demos, and choosing a deployment target.

> This fills the "Serving your model" slot in the roadmap. For the production-scale stack (Triton, TorchServe, model optimization), see **Level 6.4** in the main README.

## 1. What is Model Serving?

**Model serving** means exposing a trained model as an interface that other software can call to get predictions:

| Serving style | How it works | Example use case |
|---|---|---|
| **Batch inference** | Score large volumes offline on a schedule | Nightly churn scoring for 10M users |
| **REST / gRPC API (real-time)** | Model answers HTTP requests in milliseconds | Fraud check at checkout |
| **Stream / on-device** | Model runs on events or directly on a phone/edge device | Keyboard suggestion, offline vision app |

The typical real-time serving pipeline looks like this:

```
Client request → API server → preprocess → load model → predict → postprocess → response
```

Everything you learned in the **Pipelines** and **Data Leakage** sections matters here: the exact same preprocessing used at training time must be applied at inference time. The safest way to guarantee that is to **serve the whole `Pipeline` object, not just the model**.

## 2. Model Serialization

Before a model can be served, it must be saved to disk in a format your serving environment can load.

| Format | Library | Best for | Notes |
|---|---|---|---|
| **Pickle** | built-in | Quick Python-only experiments | Version-sensitive; never unpickle untrusted files |
| **Joblib** | `joblib` | scikit-learn models with large numpy arrays | Standard choice for sklearn; efficient array compression |
| **ONNX** | `onnx`, `onnxruntime` | Cross-framework, optimized inference | Export sklearn/PyTorch/XGBoost to a portable format |
| **TorchScript / `torch.export`** | PyTorch | Production PyTorch inference | Graph-based execution, frees you from Python overhead |
| **Safetensors** | Hugging Face | Transformer weights | Fast and safe deserialization, the HF standard |

**Rule of thumb:** joblib/pickle for internal sklearn services, ONNX when the serving runtime is not Python, and the framework's native export path for deep learning models.

In [1]:
# Train a scikit-learn pipeline and save it with joblib
import joblib
import numpy as np
from sklearn.datasets import load_iris
from sklearn.decomposition import PCA
from sklearn.linear_model import LogisticRegression
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler

X, y = load_iris(return_X_y=True)

# Serve the WHOLE pipeline: scaling + PCA + classifier travel together
pipe = Pipeline([
    ("scaler", StandardScaler()),
    ("pca", PCA(n_components=2)),
    ("clf", LogisticRegression(max_iter=1000)),
])
pipe.fit(X, y)
print("Training accuracy:", pipe.score(X, y))

# Persist to disk
joblib.dump(pipe, "iris_pipeline.joblib")
print("Saved model artifact: iris_pipeline.joblib")

Training accuracy: 0.9333333333333333
Saved model artifact: iris_pipeline.joblib


In [2]:
# Load it back and confirm predictions are identical
loaded_pipe = joblib.load("iris_pipeline.joblib")

X_new = np.array([[5.1, 3.5, 1.4, 0.2],
                  [6.7, 3.0, 5.2, 2.3]])

print("In-memory predictions: ", pipe.predict(X_new))
print("Loaded-model predictions:", loaded_pipe.predict(X_new))
print("Same probabilities:", np.allclose(pipe.predict_proba(X_new),
                                         loaded_pipe.predict_proba(X_new)))

In-memory predictions:  [0 2]
Loaded-model predictions: [0 2]
Same probabilities: True


## 3. Serving with FastAPI

**FastAPI** is the 2026 default for ML model APIs: asynchronous, automatic OpenAPI docs, and native Pydantic validation of request bodies. Flask still appears in legacy services, but new projects should start with FastAPI.

A minimal serving app (`app.py`):

```python
# app.py — run with: uvicorn app:app --reload
import joblib
import numpy as np
from fastapi import FastAPI
from pydantic import BaseModel, Field

# Load once at startup, not per request
model = joblib.load("iris_pipeline.joblib")
app = FastAPI(title="Iris Classifier API", version="1.0.0")

# Define the request contract — validation for free
class IrisFeatures(BaseModel):
    sepal_length: float = Field(ge=0, le=30)
    sepal_width: float = Field(ge=0, le=30)
    petal_length: float = Field(ge=0, le=30)
    petal_width: float = Field(ge=0, le=30)

CLASS_NAMES = ["setosa", "versicolor", "virginica"]

@app.post("/predict")
def predict(f: IrisFeatures):
    features = np.array([[f.sepal_length, f.sepal_width,
                          f.petal_length, f.petal_width]])
    pred = model.predict(features)[0]
    proba = model.predict_proba(features)[0].max()
    return {"prediction": CLASS_NAMES[int(pred)], "confidence": round(float(proba), 4)}

@app.get("/health")
def health():
    return {"status": "ok", "model_loaded": model is not None}
```

**Serving checklist for any API:** load the model **once** at startup, define request/response schemas with Pydantic, add a `/health` endpoint for load balancers, log inputs and latencies (never raw PII), and set timeouts.

We can define the same app right here in the notebook (without starting the server):

In [3]:
# The FastAPI app can be defined in a notebook cell for testing
from fastapi import FastAPI
from pydantic import BaseModel, Field

serving_app = FastAPI(title="Iris Classifier API", version="1.0.0")

class IrisFeatures(BaseModel):
    sepal_length: float = Field(ge=0, le=30)
    sepal_width: float = Field(ge=0, le=30)
    petal_length: float = Field(ge=0, le=30)
    petal_width: float = Field(ge=0, le=30)

CLASS_NAMES = ["setosa", "versicolor", "virginica"]

@serving_app.post("/predict")
def predict(f: IrisFeatures):
    import numpy as np
    x = np.array([[f.sepal_length, f.sepal_width, f.petal_length, f.petal_width]])
    pred = loaded_pipe.predict(x)[0]
    proba = loaded_pipe.predict_proba(x)[0].max()
    return {"prediction": CLASS_NAMES[int(pred)], "confidence": round(float(proba), 4)}

print("FastAPI app defined. Run `uvicorn app:app` in a terminal to serve it.")

FastAPI app defined. Run `uvicorn app:app` in a terminal to serve it.


## 4. Quick Demos with Streamlit and Gradio

For internal tools and portfolio demos, skip the API layer and use a UI framework.

**Streamlit** (`streamlit run demo.py`) — great for data apps:

```python
# demo.py
import streamlit as st
import joblib
import numpy as np

st.title("Iris Flower Classifier")
model = joblib.load("iris_pipeline.joblib")

sl = st.slider("Sepal length", 4.0, 8.0, 5.1)
sw = st.slider("Sepal width", 2.0, 4.5, 3.5)
pl = st.slider("Petal length", 1.0, 7.0, 1.4)
pw = st.slider("Petal width", 0.1, 2.5, 0.2)

if st.button("Predict"):
    pred = model.predict(np.array([[sl, sw, pl, pw]]))[0]
    st.success(f"Predicted class: {['setosa', 'versicolor', 'virginica'][int(pred)]}")
```

**Gradio** — the fastest way to put a model on the internet (and the standard on Hugging Face Spaces):

```python
import gradio as gr
import joblib
import numpy as np

model = joblib.load("iris_pipeline.joblib")
CLASS_NAMES = ["setosa", "versicolor", "virginica"]

def predict(sl, sw, pl, pw):
    p = model.predict(np.array([[sl, sw, pl, pw]]))[0]
    return CLASS_NAMES[int(p)]

demo = gr.Interface(fn=predict,
                    inputs=[gr.Slider(4, 8, value=5.1, label="Sepal length"),
                            gr.Slider(2, 4.5, value=3.5, label="Sepal width"),
                            gr.Slider(1, 7, value=1.4, label="Petal length"),
                            gr.Slider(0.1, 2.5, value=0.2, label="Petal width")],
                    outputs=gr.Label(num_top_classes=3))
demo.launch()
```

## 5. Where to Deploy

| Target | Effort | Cost model | Best for |
|---|---|---|---|
| **Local / on-prem server** | Low | Hardware you own | Internal tools, strict data privacy |
| **Docker container** | Medium | Any host that runs containers | Reproducibility — the same image runs everywhere |
| **Hugging Face Spaces** | Very low | Free tier available | Portfolio demos with Gradio/Streamlit UIs |
| **Render / Railway / Fly.io** | Low | Usage-based | Small production APIs without a cloud account |
| **AWS** | High | Pay-per-use | SageMaker endpoints, ECS/Fargate, Lambda for serverless |
| **GCP** | High | Pay-per-use | Cloud Run (containers), Vertex AI endpoints |
| **Azure** | High | Pay-per-use | Azure ML endpoints, App Service, Container Apps |

> Heroku was the classic beginner target, but its free tier was removed in 2022 — for free demo hosting today, **Hugging Face Spaces** is the better default. Full Kubernetes-based serving (with scaling, canary releases, and A/B testing) is covered in **Level 6** of this roadmap.

**Docker in one minute** — a minimal `Dockerfile` for the FastAPI service above:

```dockerfile
FROM python:3.12-slim
WORKDIR /app
COPY requirements.txt .
RUN pip install --no-cache-dir -r requirements.txt
COPY app.py iris_pipeline.joblib .
EXPOSE 8000
CMD ["uvicorn", "app:app", "--host", "0.0.0.0", "--port", "8000"]
```

## 6. Production Serving Checklist

Before calling a served model "production-ready", you should be able to answer yes to all of these:

1. **Reproducibility** — model artifact, code, and environment are versioned; the served pipeline matches training preprocessing exactly.
2. **Versioning** — every deployment records the model version, training data version, and git commit (an MLflow Model Registry makes this routine).
3. **Health & monitoring** — a `/health` endpoint, latency + error-rate metrics, and input drift tracking are in place.
4. **Logging** — requests and predictions are logged with the ability to replay them; PII is masked or excluded.
5. **Performance** — latency budget defined (p95/p99), and the model meets it under realistic load (quantization or ONNX if not).
6. **Rollback** — a previous model version can be restored in minutes, not hours.

## 7. Key Takeaways

- Serve the **whole pipeline**, not just the estimator — that is your data-leakage insurance at inference time.
- **joblib** for sklearn artifacts, **FastAPI** for the API layer, **Gradio/Streamlit** for demos, **HF Spaces** for free hosting.
- Serialization format, loading strategy (once at startup), and monitoring are the three decisions that separate a demo from a production service.
- Serving is the doorway into MLOps — everything in **Level 6** (containers, CI/CD, monitoring) extends what you built here.